# Panel de Estado de Situación de Deudores (ESD)

Apila los archivos `esd/{codigo}.txt` de los 133 dumps IEF. Cada archivo trae la cartera de financiaciones por situación del deudor (1 a 5) y por tipo (comercial / consumo / vivienda), en formato wide con 5 fechas trimestrales por columna.

Las 5 fechas del ESD están en `entidad/{cod}.txt` del mismo dump (columnas 18-22, distinta posición que las del balance).

Output: `data/interim/paneles_largos/panel_esd.parquet` en formato long, con columnas análogas a `panel_indicadores`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "utils"))
from paths import RAW, INTERIM, DIMENSIONES, PANELES, REPO

import pandas as pd
import duckdb

IEF_ROOT = RAW / "bcra_ief"
OUT = PANELES / "panel_esd.parquet"
OUT.parent.mkdir(parents=True, exist_ok=True)

ESD_FECHA_COLS = ["v1", "v2", "v3", "v4", "v5"]
# entidad/cod.txt: las 5 fechas del ESD están en cols 18-22 (1-based) = índices 17-21
ENTIDAD_FECHA_ESD_IDX = list(range(17, 22))

## Helpers

In [ ]:
def leer_fechas_esd(entidad_file):
    texto = entidad_file.read_text(encoding="ISO-8859-1")
    campos = [c.strip().strip('"') for c in texto.split("\t")]
    if len(campos) <= ENTIDAD_FECHA_ESD_IDX[-1]:
        return []
    fechas = [campos[i] for i in ENTIDAD_FECHA_ESD_IDX]
    # Las fechas vienen como "Mar-2024", "Jun-2024", etc; las paso a yyyymm
    meses = {"Ene":"01","Feb":"02","Mar":"03","Abr":"04","May":"05","Jun":"06",
             "Jul":"07","Ago":"08","Set":"09","Sep":"09","Oct":"10","Nov":"11","Dic":"12"}
    yyyymm = []
    for f in fechas:
        try:
            mes_str, yy = f.split("-")
            yyyymm.append(yy + meses[mes_str])
        except Exception:
            yyyymm.append(None)
    return yyyymm

def leer_esd(esd_file, fechas_5):
    cols = ["codigo_entidad", "nombre_entidad", "fecha_corte", "codigo_linea",
            "descripcion_situacion", "v1", "v2", "v3", "v4", "v5", "formato_valor"]
    df = pd.read_csv(esd_file, sep="\t", header=None, names=cols,
                     encoding="ISO-8859-1", dtype=str)
    for c in ESD_FECHA_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    mapeo = dict(zip(ESD_FECHA_COLS, fechas_5))
    long = df.melt(
        id_vars=["codigo_entidad", "nombre_entidad", "codigo_linea", "descripcion_situacion", "formato_valor"],
        value_vars=ESD_FECHA_COLS, var_name="slot_fecha", value_name="valor"
    )
    long["yyyymm"] = long["slot_fecha"].map(mapeo)
    long = long.drop(columns=["slot_fecha"]).dropna(subset=["yyyymm"])
    return long

## Apilado

In [ ]:
dumps = sorted([d for d in IEF_ROOT.iterdir() if d.is_dir() and d.name.isdigit()])
bloques = []
for d in dumps:
    yyyymm_dump = d.name
    esd_dir = d / "Entfin/Tec_Cont/esd"
    entidad_dir = d / "Entfin/Tec_Cont/entidad"
    if not esd_dir.exists() or not entidad_dir.exists():
        continue
    for esd_file in esd_dir.glob("*.txt"):
        if esd_file.name == "formato.txt":
            continue
        cod = esd_file.stem
        entidad_file = entidad_dir / f"{cod}.txt"
        if not entidad_file.exists():
            continue
        try:
            fechas = leer_fechas_esd(entidad_file)
            if len(fechas) != 5 or any(f is None for f in fechas):
                continue
            long = leer_esd(esd_file, fechas)
            long["dump_yyyymm"] = int(yyyymm_dump)
            bloques.append(long)
        except Exception:
            pass

raw = pd.concat(bloques, ignore_index=True)
print(f"Filas apiladas: {len(raw):,}")

## Deduplicación y persistencia

In [ ]:
raw = raw[raw["yyyymm"].astype(str).str.match(r"^\d{6}$")].copy()
raw["yyyymm"] = raw["yyyymm"].astype(int)
raw = raw.sort_values(["codigo_entidad", "yyyymm", "codigo_linea", "dump_yyyymm"])
panel = raw.drop_duplicates(subset=["codigo_entidad", "yyyymm", "codigo_linea"], keep="last")
panel.to_parquet(OUT, index=False)
print(f"Escrito: {OUT.relative_to(REPO)}")
print(f"Filas: {len(panel):,}")

## Validaciones

In [ ]:
assert panel[["codigo_entidad", "yyyymm", "codigo_linea"]].duplicated().sum() == 0
print("Validaciones OK")
print(f"  entidades: {panel['codigo_entidad'].nunique()}")
print(f"  códigos de línea: {panel['codigo_linea'].nunique()}")
print(f"  rango fechas: {panel['yyyymm'].min()} - {panel['yyyymm'].max()}")

## Muestra

In [ ]:
duckdb.sql(f"""
    select codigo_entidad, yyyymm, descripcion_situacion, valor
    from '{OUT}'
    where codigo_entidad = '00007' and yyyymm = (select max(yyyymm) from '{OUT}')
    limit 10
""").df()